In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import torch
import random
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from datasets import Dataset
from rank_bm25 import BM25Okapi
from transformers import EarlyStoppingCallback
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    util,
)
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.losses import MultipleNegativesRankingLoss, CosineSimilarityLoss
from sentence_transformers.evaluation import TripletEvaluator, EmbeddingSimilarityEvaluator
from sentence_transformers.similarity_functions import SimilarityFunction
import kagglehub
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
try:
    torch.cuda.empty_cache()
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilisation du device : {device}")
if device == "cuda":
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. TÉLÉCHARGEMENT ET PRÉPARATION DE BASE

path = kagglehub.dataset_download("kanchana1990/real-estate-data-london-2024")
csv_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.read_csv(csv_files[0])

def clean_html(html_text):
    if not isinstance(html_text, str): return ""
    return BeautifulSoup(html_text, "html.parser").get_text(separator=" ").strip()

print("Nettoyage des données en cours...")
df['clean_description'] = df['descriptionHtml'].apply(clean_html)

cols_to_fill = ['title', 'propertyType', 'sizeSqFeetMax', 'bedrooms', 'bathrooms', 'price']
for col in cols_to_fill:
    df[col] = df[col].fillna("N/A").astype(str)

df['bedrooms']  = df['bedrooms'].apply(lambda x: x.replace('.0', '') if x.endswith('.0') else x)
df['bathrooms'] = df['bathrooms'].apply(lambda x: x.replace('.0', '') if x.endswith('.0') else x)

df['rich_anchor'] = (
    df['title'] +
    " [ATTR] Type: "  + df['propertyType'] +
    " [ATTR] Beds: "  + df['bedrooms'] +
    " [ATTR] Baths: " + df['bathrooms'] +
    " [ATTR] Size: "  + df['sizeSqFeetMax'] + " sqft" +
    " [ATTR] Price: " + df['price']
)

df_ready = df[['rich_anchor', 'clean_description']].rename(
    columns={'rich_anchor': 'anchor', 'clean_description': 'positive'}
)

# 80% Train / 20% Test
df_train, df_test = train_test_split(df_ready, test_size=0.2, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train : {len(df_train)} exemples | Test : {len(df_test)} exemples")

In [ ]:
# 2. AUGSBERT : DISTILLATION AVEC CROSS-ENCODER + BM25

print("\n[AUGSBERT] Chargement du Modèle Professeur (Cross-Encoder)...")
ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device, max_length=512)

def generate_augsbert_triplets(df_split, is_train=True):
    corpus           = df_split['positive'].tolist()
    tokenized_corpus = [doc.lower().split() for doc in corpus]
    bm25             = BM25Okapi(tokenized_corpus)

    augmented_data = []

    print(f"Génération des triplets ({'Train' if is_train else 'Test'})...")
    for i, row in tqdm(df_split.iterrows(), total=len(df_split)):
        anchor        = row['anchor']
        true_positive = row['positive']

        # 1. BM25 : top 20 descriptions lexicalement proches
        query       = anchor.lower().split()
        bm25_scores = bm25.get_scores(query)
        top_20_idx  = np.argsort(bm25_scores)[::-1][:20]

        # 2. Préparation des paires pour le Cross-Encoder
        pairs_to_score = [[anchor, corpus[idx]] for idx in top_20_idx]

        # 3. Le Professeur note ces 20 paires
        ce_scores = ce_model.predict(pairs_to_score)

        # Association description <-> score CE (en excluant le vrai positif)
        scored_candidates = []
        for idx, score in zip(top_20_idx, ce_scores):
            if idx != i:
                scored_candidates.append((corpus[idx], score))

        scored_candidates.sort(key=lambda x: x[1], reverse=True)

        if len(scored_candidates) == 0:
            continue

        # ✅ 1 seul triplet par anchor : hard negative = le pire selon le CE
        best_hard_negative = scored_candidates[-1][0]

        augmented_data.append({
            'anchor':   anchor,
            'positive': true_positive,
            'negative': best_hard_negative
        })

    return pd.DataFrame(augmented_data)

print("\n--- DÉMARRAGE DE L'AUGMENTATION AUGSBERT ---")
df_train_aug = generate_augsbert_triplets(df_train, is_train=True)
df_test_aug  = generate_augsbert_triplets(df_test,  is_train=False)

print(f"\nTrain : {len(df_train_aug)} triplets | Test : {len(df_test_aug)} triplets")

# Libération VRAM
del ce_model
torch.cuda.empty_cache()

# Conversion en datasets Hugging Face
train_dataset = Dataset.from_pandas(df_train_aug).select_columns(["anchor", "positive", "negative"])
test_dataset  = Dataset.from_pandas(df_test_aug).select_columns(["anchor", "positive", "negative"])

def add_special_tokens_to_text(example):
    return {
        "anchor":   "[TITLE] " + example["anchor"],
        "positive": "[DESC] "  + example["positive"],
        "negative": "[DESC] "  + example["negative"]
    }

train_dataset = train_dataset.map(add_special_tokens_to_text)
test_dataset  = test_dataset.map(add_special_tokens_to_text)

print("Datasets prêts.")

In [ ]:
# 3. CHARGEMENT DU MODÈLE DE BASE (BGE)

print("\n[ÉLÈVE] Chargement du modèle BGE-base-en-v1.5...")
model_name = 'BAAI/bge-base-en-v1.5'
model = SentenceTransformer(model_name, device=device)

model.max_seq_length = 512

word_embedding_model = model._first_module()
tokenizer            = word_embedding_model.tokenizer
transformer_model    = word_embedding_model.auto_model

# Ajout des tokens spéciaux
special_tokens     = ['[TITLE]', '[DESC]', '[ATTR]']
added_tokens_count = tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})

if added_tokens_count > 0:
    transformer_model.resize_token_embeddings(len(tokenizer))
    embedding_layer = transformer_model.get_input_embeddings()
    new_token_ids   = tokenizer.convert_tokens_to_ids(special_tokens)

    initialization_texts = ['property title', 'property description', 'property attribute']
    for token_id, init_text in zip(new_token_ids, initialization_texts):
        init_token_ids = tokenizer(init_text, add_special_tokens=False, return_tensors='pt')['input_ids']
        init_token_ids = init_token_ids.to(device)
        with torch.no_grad():
            mean_embed = embedding_layer(init_token_ids).squeeze(0).mean(dim=0)
            embedding_layer.weight.data[token_id] = mean_embed

print("Tokens spéciaux ajoutés et initialisés.")

# Vérification zero-shot avant entraînement
print("\n[ZERO-SHOT CHECK] Scores avant entraînement...")
sample_anchor   = "[TITLE] " + df_test_aug['anchor'].iloc[0]
sample_positive = "[DESC] "  + df_test_aug['positive'].iloc[0]
sample_negative = "[DESC] "  + df_test_aug['negative'].iloc[0]

emb     = model.encode([sample_anchor, sample_positive, sample_negative], normalize_embeddings=True)
sim_pos = float(np.dot(emb[0], emb[1]))
sim_neg = float(np.dot(emb[0], emb[2]))
print(f"  Similarité anchor/positif : {sim_pos:.4f}")
print(f"  Similarité anchor/négatif : {sim_neg:.4f}")
print(f"  ✅ Correct (pos > neg) : {sim_pos > sim_neg}")

In [ ]:
# 4. GEL DES COUCHES

print("Configuration du gel des couches...")
auto_model = model._first_module().auto_model

# Tout geler
for param in auto_model.parameters():
    param.requires_grad = False

# Dégeler les 4 dernières couches + pooler
# BGE-base a 12 couches (layer.8 à layer.11 = 4 dernières)
for name, param in auto_model.named_parameters():
    if any(layer in name for layer in ["layer.8", "layer.9", "layer.10", "layer.11", "pooler"]):
        param.requires_grad = True

# Dégeler les embeddings car on a ajouté des tokens spéciaux
for param in auto_model.embeddings.parameters():
    param.requires_grad = True

total_params     = sum(p.numel() for p in auto_model.parameters())
trainable_params = sum(p.numel() for p in auto_model.parameters() if p.requires_grad)
print(f"Paramètres totaux      : {total_params:,}")
print(f"Paramètres entraînables: {trainable_params:,} ({100 * trainable_params / total_params:.1f}%)")

In [ ]:
# 5. ENTRAÎNEMENT PHASE 1 : DISCRIMINATION (MNRL scale=50)

OUTPUT_DIR = "output/bge-augsbert-london"

# ✅ Scale=50 pour une séparation plus nette dès la phase 1
loss_phase1 = MultipleNegativesRankingLoss(model, scale=50.0)

evaluator_phase1 = TripletEvaluator(
    anchors=df_test_aug['anchor'].apply(lambda x: "[TITLE] " + x).tolist(),
    positives=df_test_aug['positive'].apply(lambda x: "[DESC] " + x).tolist(),
    negatives=df_test_aug['negative'].apply(lambda x: "[DESC] " + x).tolist(),
    name="test-triplet",
)

print("\n[BASELINE] Score zero-shot du modèle non fine-tuné :")
evaluator_phase1(model)

args_phase1 = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR + "/phase1",
    report_to="none",
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,
    bf16=False,
    dataloader_num_workers=4,

    warmup_ratio=0.1,
    eval_strategy='steps',
    eval_steps=25,
    save_strategy='steps',
    save_steps=25,
    logging_steps=5,

    max_grad_norm=1.0,
    lr_scheduler_type='cosine',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=42,
    remove_unused_columns=False,
)

trainer_phase1 = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss=loss_phase1,
    args=args_phase1,
    evaluator=evaluator_phase1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("\n[PHASE 1] Discrimination (MNRL scale=50)...")
trainer_phase1.train()

In [ ]:
# 6. PRÉPARATION DU DATASET PHASE 2 : CALIBRATION

def build_calibration_dataset(df_aug, label_positive=1.0, label_negative=0.05):
    """
    Crée des paires (sentence1, sentence2, label) pour CosineSimilarityLoss.
    label=1.0  → paire positive, score cible = 1.0
    label=0.05 → paire négative, score cible proche de 0.0
    (0.05 plutôt que 0.0 pour éviter collapse géométrique)
    """
    pairs = []
    for _, row in df_aug.iterrows():
        anchor   = "[TITLE] " + row['anchor']
        positive = "[DESC] "  + row['positive']
        negative = "[DESC] "  + row['negative']

        pairs.append({"sentence1": anchor, "sentence2": positive, "label": label_positive})
        pairs.append({"sentence1": anchor, "sentence2": negative, "label": label_negative})

    return Dataset.from_list(pairs)

train_dataset_phase2 = build_calibration_dataset(df_train_aug)
test_dataset_phase2  = build_calibration_dataset(df_test_aug)

labels = train_dataset_phase2['label']
print(f"Labels Phase 2 → Positifs (1.0) : {labels.count(1.0)} | Négatifs (0.05) : {labels.count(0.05)}")

In [ ]:
# 7. ENTRAÎNEMENT PHASE 2 : CALIBRATION (CosineSimilarityLoss)

loss_phase2 = CosineSimilarityLoss(model)

evaluator_phase2 = EmbeddingSimilarityEvaluator(
    sentences1=test_dataset_phase2['sentence1'],
    sentences2=test_dataset_phase2['sentence2'],
    scores=test_dataset_phase2['label'],
    name="calibration-eval",
    # ✅ similarity_fct supprimé (non supporté dans cette version)
)

args_phase2 = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR + "/phase2",
    report_to="none",
    num_train_epochs=3,
    learning_rate=5e-6,
    weight_decay=0.0,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,
    bf16=False,
    dataloader_num_workers=4,

    warmup_ratio=0.05,
    eval_strategy='steps',
    eval_steps=20,
    save_strategy='steps',
    save_steps=20,
    logging_steps=5,

    max_grad_norm=0.5,
    lr_scheduler_type='cosine',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=42,
    remove_unused_columns=False,
)

trainer_phase2 = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset_phase2,
    eval_dataset=test_dataset_phase2,
    loss=loss_phase2,
    args=args_phase2,
    evaluator=evaluator_phase2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

print("\n[PHASE 2] Calibration des scores (CosineSimilarityLoss)...")
trainer_phase2.train()

In [ ]:
# 8. VÉRIFICATION DE LA CALIBRATION

print("\n=== VÉRIFICATION DES SCORES APRÈS CALIBRATION ===\n")

n_samples = 10
anchors   = df_test_aug['anchor'].tolist()[:n_samples]
positives = df_test_aug['positive'].tolist()[:n_samples]
negatives = df_test_aug['negative'].tolist()[:n_samples]

emb_anchors   = model.encode(["[TITLE] " + a for a in anchors],   normalize_embeddings=True)
emb_positives = model.encode(["[DESC] "  + p for p in positives], normalize_embeddings=True)
emb_negatives = model.encode(["[DESC] "  + n for n in negatives], normalize_embeddings=True)

pos_scores = [float(np.dot(emb_anchors[i], emb_positives[i])) for i in range(n_samples)]
neg_scores = [float(np.dot(emb_anchors[i], emb_negatives[i])) for i in range(n_samples)]

print(f"{'Exemple':<10} {'Score Positif':>15} {'Score Négatif':>15} {'Écart':>10} {'OK':>5}")
print("-" * 60)
for i in range(n_samples):
    ecart = pos_scores[i] - neg_scores[i]
    ok    = "✅" if pos_scores[i] > 0.9 and neg_scores[i] < 0.2 else "⚠️"
    print(f"{i:<10} {pos_scores[i]:>15.4f} {neg_scores[i]:>15.4f} {ecart:>10.4f} {ok:>5}")

print(f"\n{'Moyenne':<10} {np.mean(pos_scores):>15.4f} {np.mean(neg_scores):>15.4f} "
      f"{np.mean(pos_scores) - np.mean(neg_scores):>10.4f}")

In [ ]:
# 9. SAUVEGARDE DU MODÈLE FINAL

final_model_path = os.path.join(OUTPUT_DIR, "final_model")
model.save(final_model_path)
print(f"\nModèle final sauvegardé dans : {final_model_path}")

In [ ]:
# 10. GRAPHIQUES DES DEUX PHASES

def plot_training_history(trainer, title="Training History"):
    history = trainer.state.log_history

    train_steps, train_loss = [], []
    eval_steps,  eval_loss  = [], []

    for entry in history:
        if 'loss' in entry and 'eval_loss' not in entry:
            train_steps.append(entry['step'])
            train_loss.append(entry['loss'])
        elif 'eval_loss' in entry:
            eval_steps.append(entry['step'])
            eval_loss.append(entry['eval_loss'])

    plt.figure(figsize=(12, 6))
    plt.plot(train_steps, train_loss, label='Training Loss',   color='blue', alpha=0.6)

    if eval_steps:
        plt.plot(eval_steps, eval_loss, label='Validation Loss', color='red',
                 linewidth=2, marker='o')

    if eval_loss:
        best_idx  = int(np.argmin(eval_loss))
        best_step = eval_steps[best_idx]
        best_val  = eval_loss[best_idx]
        plt.axvline(x=best_step, color='green', linestyle='--', alpha=0.7,
                    label=f'Best checkpoint (step {best_step}, loss={best_val:.4f})')

    plt.title(f"Evolution of Loss — {title}")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

plot_training_history(trainer_phase1, title="Phase 1 — Discrimination (MNRL scale=50)")
plot_training_history(trainer_phase2, title="Phase 2 — Calibration (CosineSimilarityLoss)")

In [ ]:
model = SentenceTransformer(final_model_path)

df_test_complet = test_dataset.to_pandas()

# On prend 5 exemples au hasard
sample_df = df_test_complet.sample(5, random_state=42).reset_index(drop=True)
# Application des tokens spéciaux de contexte (comme pour l'entraînement)
anchors = ["[TITLE] " + a for a in sample_df['anchor']]
positives = ["[DESC] " + p for p in sample_df['positive']]
negatives = ["[DESC] " + n for n in sample_df['negative']]

print("\nCalcul des vecteurs (embeddings) en cours...")
emb_anchors = model.encode(anchors, convert_to_tensor=True)
emb_positives = model.encode(positives, convert_to_tensor=True)
emb_negatives = model.encode(negatives, convert_to_tensor=True)

print("\n" + "="*60)
print(" RÉSULTATS DU CRASH-TEST SÉMANTIQUE")
print("="*60)

succes = 0

for i in range(len(anchors)):
    # Calcul de la similarité cosinus entre l'ancre et les deux descriptions
    score_pos = util.cos_sim(emb_anchors[i], emb_positives[i]).item()
    score_neg = util.cos_sim(emb_anchors[i], emb_negatives[i]).item()
    
    # Nettoyage de l'affichage (on tronque les descriptions trop longues pour la lisibilité)
    desc_pos_courte = positives[i][:150].replace('\n', ' ') + "..."
    desc_neg_courte = negatives[i][:150].replace('\n', ' ') + "..."
    
    print(f"\n🔍 EXEMPLE {i+1}")
    print(f"ANCRE : {anchors[i]}")
    print(f"VRAIE ANNONCE (Score: {score_pos:.4f}) : {desc_pos_courte}")
    print(f"PIÈGE BM25   (Score: {score_neg:.4f}) : {desc_neg_courte}")
    
    if score_pos > score_neg:
        print("Verdict : SUCCÈS (Le modèle a préféré la bonne annonce)")
        succes += 1
    else:
        print("Verdict : ÉCHEC (Le modèle s'est fait avoir par le piège lexical)")
        
    print("-" * 60)

# Résultat global
print(f"\nScore final sur cet échantillon : {succes}/{len(anchors)} succès.")
print(f"Taux de réussite face aux Hard Negatives : {(succes/len(anchors))*100:.0f}%\n")